# BuildMate Rentals - Microsoft Fabric end to end
## Step 1 - Bronze, land it exactly as it arrived

**Attach this notebook to the `lh_buildmate` Lakehouse before running** (Explorer pane on the left, Add Lakehouse).

**The Bronze rule: Bronze is a photograph, not a repair job.**

Every column is read as a string. On purpose. If you let Spark guess types on the way in, `Rs. 42,180.00` becomes null before you ever see it and you never learn how many rows you lost. Land it raw, cast in Silver, and count the failures where you can see them.

The only columns we add are `ingested_at` and `source_file`.

In [1]:
from pyspark.sql import functions as F, Window

RAW = "Files/buildmate_raw_data/raw"

def to_bronze(df):
    """Add lineage columns. Nothing else. No cleaning happens in Bronze."""
    return (df.withColumn("ingested_at", F.current_timestamp())
              .withColumn("source_file", F.input_file_name()))

def read_raw_csv(path):
    # inferSchema is OFF deliberately. See the note above.
    return (spark.read
            .option("header", True)
            .option("inferSchema", False)
            .csv(path))


StatementMeta(, fc01572c-6ff2-4125-9d5a-dc14d3780027, 3, Finished, Available, Finished, False)

### Land the four batch sources

Note the wildcard on rentals: `rentals_*.csv` picks up all 31 files in one read.

In [2]:
to_bronze(read_raw_csv(f"{RAW}/customer_master/customer_master_export.csv")) \
    .write.mode("overwrite").saveAsTable("bronze_customers")

to_bronze(read_raw_csv(f"{RAW}/depots/depot_master.csv")) \
    .write.mode("overwrite").saveAsTable("bronze_depots")

# 31 files, one read: 30 daily drops plus the resend
to_bronze(read_raw_csv(f"{RAW}/rentals/rentals_*.csv")) \
    .write.mode("overwrite").saveAsTable("bronze_rentals")

to_bronze(read_raw_csv(f"{RAW}/billing/billing_export.csv")) \
    .write.mode("overwrite").saveAsTable("bronze_billing")

for t in ["bronze_customers", "bronze_depots", "bronze_rentals", "bronze_billing"]:
    print(f"{t:20s} {spark.table(t).count():>6,}")

StatementMeta(, fc01572c-6ff2-4125-9d5a-dc14d3780027, 4, Finished, Available, Finished, False)

bronze_customers        602
bronze_depots             6
bronze_rentals          976
bronze_billing          787


### Prove the Bronze contract

Every column a string, and both lineage columns on every table. If either check fails, something cleaned or recast the data.

In [3]:
for t in ["bronze_customers", "bronze_depots", "bronze_rentals", "bronze_billing"]:
    df = spark.table(t)
    non_string = [f.name for f in df.schema.fields if f.dataType.simpleString() != "string" and f.name != "ingested_at"]
    assert not non_string, f"{t}: non-string columns {non_string}"
    assert {"ingested_at", "source_file"} <= set(df.columns), f"{t}: lineage columns missing"
    print(f"{t:20s} all columns string (ingested_at is the lineage timestamp), lineage present")

StatementMeta(, fc01572c-6ff2-4125-9d5a-dc14d3780027, 5, Finished, Available, Finished, False)

bronze_customers     all columns string (ingested_at is the lineage timestamp), lineage present
bronze_depots        all columns string (ingested_at is the lineage timestamp), lineage present
bronze_rentals       all columns string (ingested_at is the lineage timestamp), lineage present
bronze_billing       all columns string (ingested_at is the lineage timestamp), lineage present


### The source-file check for 10 June returns two files with the same row count

In [5]:
# Bronze keeps the evidence. Prove the resend is real, and which file each row came from.
(spark.table("bronze_rentals")
   .groupBy(F.regexp_extract("source_file", r"([^/]+)$", 1).alias("file"))
   .count()
   .filter(F.col("file").contains("2026-06-10"))
   .show(truncate=False))

StatementMeta(, fc01572c-6ff2-4125-9d5a-dc14d3780027, 7, Finished, Available, Finished, False)

+----------------------------------------------------------------+-----+
|file                                                            |count|
+----------------------------------------------------------------+-----+
|rentals_2026-06-10_resend.csv?version=1791568301000?flength=2465|31   |
|rentals_2026-06-10.csv?version=1791568301000?flength=2465       |31   |
+----------------------------------------------------------------+-----+

